# DSA 8302 Practical Exercise 2
## Miraa Transportation Problem: VAM Verification and Optimal Solution
**Strathmore Institute of Mathematical Sciences | Masters in Data Science**

This notebook walks through the transportation problem step by step.
It first encodes the problem data exactly as stated in the group report,
then replicates the VAM procedure in code, and finally uses PuLP (a linear
programming library) to find the mathematically optimal solution so both
results can be compared side by side.


## 1. Setup: Import Libraries

In [2]:
import numpy as np
import pandas as pd
from pulp import (
    LpProblem, LpVariable, LpMinimize, lpSum, value, LpStatus, PULP_CBC_CMD
)

# Suppress PuLP solver output so the notebook stays readable
import logging
logging.getLogger("pulp").setLevel(logging.CRITICAL)

print("All libraries loaded successfully.")


All libraries loaded successfully.


## 2. The Problem Data

The three growing areas in Meru County supply fresh miraa to four Kenyan markets.
Every cost figure below comes directly from the group report; they are
perishability-adjusted composite costs in KES per kilogram.

| Route | Base | Road | Vehicle | Urgency | Total |
|-------|------|------|---------|---------|-------|
| Maua to Isiolo | 2.2 | 3.5 | 0.0 | 4.3 | **10** |
| Maua to Kilifi | 19.2 | 0.5 | 1.0 | 0.3 | **21** |
| Maua to Eastleigh | 12.8 | 0.0 | 0.5 | 4.2 | **18** |
| Maua to Likoni | 21.2 | 0.5 | 1.0 | 0.2 | **23** |
| Laare to Isiolo | 3.0 | 3.5 | 0.0 | 4.3 | **11** |
| Laare to Kilifi | 20.0 | 0.5 | 1.0 | 0.3 | **22** |
| Laare to Eastleigh | 11.6 | 0.0 | 0.5 | 4.2 | **16** |
| Laare to Likoni | 21.6 | 0.5 | 1.0 | 0.2 | **23** |
| Kangeta to Isiolo | 2.6 | 3.5 | 0.0 | 4.3 | **10** |
| Kangeta to Kilifi | 19.6 | 0.5 | 1.0 | 0.3 | **21** |
| Kangeta to Eastleigh | 12.2 | 0.0 | 0.5 | 4.2 | **17** |
| Kangeta to Likoni | 20.8 | 0.5 | 1.0 | 0.2 | **23** |


In [3]:
# Sources and their supply quantities (kg)
sources = ["Maua (S1)", "Laare (S2)", "Kangeta (S3)"]
supply  = [2500, 1800, 1700]

# Destinations and their demand quantities (kg)
destinations = ["Isiolo (D1)", "Kilifi (D2)", "Eastleigh (D3)", "Likoni (D4)"]
demand        = [1500, 1500, 2000, 1000]

# Cost matrix: rows = sources, columns = destinations (KES/kg)
cost = np.array([
    [10, 21, 18, 23],   # Maua
    [11, 22, 16, 23],   # Laare
    [10, 21, 17, 23],   # Kangeta
])

# Display as a readable DataFrame
cost_df = pd.DataFrame(cost, index=sources, columns=destinations)
cost_df["Supply (kg)"] = supply

demand_row = pd.DataFrame(
    [demand + [sum(demand)]],
    index=["Demand (kg)"],
    columns=destinations + ["Supply (kg)"]
)

display_df = pd.concat([cost_df, demand_row])
print("Transportation Cost Matrix (KES/kg)\n")
print(display_df.to_string())
print()
print(f"Total Supply : {sum(supply):,} kg")
print(f"Total Demand : {sum(demand):,} kg")
print(f"Balanced     : {sum(supply) == sum(demand)}")


Transportation Cost Matrix (KES/kg)

              Isiolo (D1)  Kilifi (D2)  Eastleigh (D3)  Likoni (D4)  Supply (kg)
Maua (S1)              10           21              18           23         2500
Laare (S2)             11           22              16           23         1800
Kangeta (S3)           10           21              17           23         1700
Demand (kg)          1500         1500            2000         1000         6000

Total Supply : 6,000 kg
Total Demand : 6,000 kg
Balanced     : True


## 3. Vogel's Approximation Method (VAM) in Python

The function below mirrors the manual iterations in the group report exactly.
At each step it prints the active table, the row and column penalties, and the
allocation decision so you can trace the logic iteration by iteration.

VAM rule: compute the penalty for every active row and column as the difference
between the two smallest costs in that row or column. Allocate as much as
possible to the cell with the lowest cost in the row or column that has the
largest penalty. Repeat until all supply and demand are exhausted.


In [4]:
def run_vam(cost_matrix, supply_list, demand_list, sources, destinations):
    """Run VAM and return the allocation matrix and total cost."""
    m, n   = cost_matrix.shape
    supply = supply_list.copy()
    demand = demand_list.copy()
    alloc  = np.zeros((m, n), dtype=int)

    # Track which rows and columns are still active
    row_active = [True] * m
    col_active = [True] * n

    iteration = 0

    while any(supply[i] > 0 for i in range(m)) and any(demand[j] > 0 for j in range(n)):
        iteration += 1
        print(f"{'='*60}")
        print(f"Iteration {iteration}")
        print(f"{'='*60}")

        # Show the reduced cost table for active rows and columns
        active_rows = [i for i in range(m) if row_active[i] and supply[i] > 0]
        active_cols = [j for j in range(n) if col_active[j] and demand[j] > 0]

        tbl_index = [sources[i] for i in active_rows]
        tbl_cols  = [destinations[j] for j in active_cols]
        tbl_data  = cost_matrix[np.ix_(active_rows, active_cols)]
        tbl_df    = pd.DataFrame(tbl_data, index=tbl_index, columns=tbl_cols)
        print("Active cost table:")
        print(tbl_df.to_string())
        print()

        # Row penalties
        row_penalties = {}
        for i in active_rows:
            costs = sorted(cost_matrix[i, j] for j in active_cols)
            penalty = costs[1] - costs[0] if len(costs) >= 2 else costs[0]
            row_penalties[i] = penalty

        # Column penalties
        col_penalties = {}
        for j in active_cols:
            costs = sorted(cost_matrix[i, j] for i in active_rows)
            penalty = costs[1] - costs[0] if len(costs) >= 2 else costs[0]
            col_penalties[j] = penalty

        # Print penalties
        print("Row penalties:", {sources[i]: v for i, v in row_penalties.items()})
        print("Col penalties:", {destinations[j]: v for j, v in col_penalties.items()})

        # Find the maximum penalty across all rows and columns
        all_penalties = {("row", i): v for i, v in row_penalties.items()}
        all_penalties.update({("col", j): v for j, v in col_penalties.items()})
        max_penalty_key = max(all_penalties, key=all_penalties.get)
        max_penalty_val = all_penalties[max_penalty_key]

        if max_penalty_key[0] == "row":
            i_star = max_penalty_key[1]
            # Lowest cost column in this row
            j_star = min(active_cols, key=lambda j: cost_matrix[i_star, j])
            print(f"\nMax penalty = {max_penalty_val} at row {sources[i_star]}")
            print(f"Lowest cost in that row: {cost_matrix[i_star, j_star]} at {destinations[j_star]}")
        else:
            j_star = max_penalty_key[1]
            # Lowest cost row in this column
            i_star = min(active_rows, key=lambda i: cost_matrix[i, j_star])
            print(f"\nMax penalty = {max_penalty_val} at col {destinations[j_star]}")
            print(f"Lowest cost in that column: {cost_matrix[i_star, j_star]} at {sources[i_star]}")

        # Allocate
        qty = min(supply[i_star], demand[j_star])
        alloc[i_star, j_star] += qty
        supply[i_star] -= qty
        demand[j_star]  -= qty

        print(f"Allocate {qty:,} kg to ({sources[i_star]}, {destinations[j_star]})")

        if supply[i_star] == 0:
            row_active[i_star] = False
            print(f"  -> {sources[i_star]} supply exhausted.")
        if demand[j_star] == 0:
            col_active[j_star] = False
            print(f"  -> {destinations[j_star]} demand satisfied.")
        print()

    # Total cost
    total_cost = int(np.sum(alloc * cost_matrix))
    return alloc, total_cost

vam_alloc, vam_cost = run_vam(cost, supply.copy(), demand.copy(), sources, destinations)


Iteration 1
Active cost table:
              Isiolo (D1)  Kilifi (D2)  Eastleigh (D3)  Likoni (D4)
Maua (S1)              10           21              18           23
Laare (S2)             11           22              16           23
Kangeta (S3)           10           21              17           23

Row penalties: {'Maua (S1)': np.int64(8), 'Laare (S2)': np.int64(5), 'Kangeta (S3)': np.int64(7)}
Col penalties: {'Isiolo (D1)': np.int64(0), 'Kilifi (D2)': np.int64(0), 'Eastleigh (D3)': np.int64(1), 'Likoni (D4)': np.int64(0)}

Max penalty = 8 at row Maua (S1)
Lowest cost in that row: 10 at Isiolo (D1)
Allocate 1,500 kg to (Maua (S1), Isiolo (D1))
  -> Isiolo (D1) demand satisfied.

Iteration 2
Active cost table:
              Kilifi (D2)  Eastleigh (D3)  Likoni (D4)
Maua (S1)              21              18           23
Laare (S2)             22              16           23
Kangeta (S3)           21              17           23

Row penalties: {'Maua (S1)': np.int64(3), 'Laare (S2)': 

## 4. VAM Final Allocation Table and Cost

In [5]:
vam_df = pd.DataFrame(vam_alloc, index=sources, columns=destinations)
vam_df["Supply Used"] = vam_alloc.sum(axis=1)
demand_check = list(vam_alloc.sum(axis=0)) + [vam_alloc.sum()]
vam_df.loc["Demand Met"] = demand_check

print("VAM Final Allocation (kg)\n")
print(vam_df.to_string())
print()

# Cost breakdown
print("Cost Breakdown")
print(f"{'Route':<35} {'Qty':>8} {'Unit Cost':>12} {'Sub-Total':>12}")
print("-" * 70)
running = 0
for i, src in enumerate(sources):
    for j, dst in enumerate(destinations):
        qty = vam_alloc[i, j]
        if qty > 0:
            sub = qty * cost[i, j]
            running += sub
            print(f"{src} -> {dst:<20} {qty:>8,} {cost[i,j]:>12} {sub:>12,}")
print("-" * 70)
print(f"{'TOTAL VAM COST (KES)':<47} {vam_cost:>12,}")


VAM Final Allocation (kg)

              Isiolo (D1)  Kilifi (D2)  Eastleigh (D3)  Likoni (D4)  Supply Used
Maua (S1)            1500         1000               0            0         2500
Laare (S2)              0            0            1800            0         1800
Kangeta (S3)            0          500             200         1000         1700
Demand Met           1500         1500            2000         1000         6000

Cost Breakdown
Route                                    Qty    Unit Cost    Sub-Total
----------------------------------------------------------------------
Maua (S1) -> Isiolo (D1)             1,500           10       15,000
Maua (S1) -> Kilifi (D2)             1,000           21       21,000
Laare (S2) -> Eastleigh (D3)          1,800           16       28,800
Kangeta (S3) -> Kilifi (D2)               500           21       10,500
Kangeta (S3) -> Eastleigh (D3)            200           17        3,400
Kangeta (S3) -> Likoni (D4)             1,000           23

## 5. Optimal Solution via Linear Programming (PuLP)

We now formulate the same problem as a Linear Programme and solve it with
PuLP's CBC solver. If the LP optimum equals the VAM cost, VAM produced the
optimal schedule. If it is lower, the LP solution is better and we can see
exactly where VAM diverged.

Formulation:

- **Decision variables**: x[i][j] = kg shipped from source i to destination j
- **Objective**: minimise sum of c[i][j] * x[i][j]
- **Supply constraints**: for each source i, sum over j of x[i][j] = supply[i]
- **Demand constraints**: for each destination j, sum over i of x[i][j] = demand[j]
- **Non-negativity**: x[i][j] >= 0


In [6]:
m = len(sources)
n = len(destinations)

prob = LpProblem("Miraa_Transport", LpMinimize)

# Decision variables
x = [[LpVariable(f"x_{i}_{j}", lowBound=0) for j in range(n)] for i in range(m)]

# Objective function
prob += lpSum(cost[i][j] * x[i][j] for i in range(m) for j in range(n))

# Supply constraints
for i in range(m):
    prob += lpSum(x[i][j] for j in range(n)) == supply[i], f"Supply_{i}"

# Demand constraints
for j in range(n):
    prob += lpSum(x[i][j] for i in range(m)) == demand[j], f"Demand_{j}"

# Solve
status = prob.solve(PULP_CBC_CMD(msg=0))
print(f"Solver status : {LpStatus[prob.status]}")
print(f"Optimal cost  : KES {int(value(prob.objective)):,}")


Solver status : Optimal
Optimal cost  : KES 101,700


## 6. LP Optimal Allocation Table

In [7]:
lp_alloc = np.array([[int(round(value(x[i][j]))) for j in range(n)] for i in range(m)])
lp_cost  = int(np.sum(lp_alloc * cost))

lp_df = pd.DataFrame(lp_alloc, index=sources, columns=destinations)
lp_df["Supply Used"] = lp_alloc.sum(axis=1)
lp_df.loc["Demand Met"] = list(lp_alloc.sum(axis=0)) + [lp_alloc.sum()]

print("LP Optimal Allocation (kg)\n")
print(lp_df.to_string())
print()

print("Cost Breakdown (LP)")
print(f"{'Route':<35} {'Qty':>8} {'Unit Cost':>12} {'Sub-Total':>12}")
print("-" * 70)
for i, src in enumerate(sources):
    for j, dst in enumerate(destinations):
        qty = lp_alloc[i, j]
        if qty > 0:
            sub = qty * cost[i, j]
            print(f"{src} -> {dst:<20} {qty:>8,} {cost[i,j]:>12} {sub:>12,}")
print("-" * 70)
print(f"{'TOTAL LP COST (KES)':<47} {lp_cost:>12,}")


LP Optimal Allocation (kg)

              Isiolo (D1)  Kilifi (D2)  Eastleigh (D3)  Likoni (D4)  Supply Used
Maua (S1)               0         1500               0         1000         2500
Laare (S2)              0            0            1800            0         1800
Kangeta (S3)         1500            0             200            0         1700
Demand Met           1500         1500            2000         1000         6000

Cost Breakdown (LP)
Route                                    Qty    Unit Cost    Sub-Total
----------------------------------------------------------------------
Maua (S1) -> Kilifi (D2)             1,500           21       31,500
Maua (S1) -> Likoni (D4)             1,000           23       23,000
Laare (S2) -> Eastleigh (D3)          1,800           16       28,800
Kangeta (S3) -> Isiolo (D1)             1,500           10       15,000
Kangeta (S3) -> Eastleigh (D3)            200           17        3,400
----------------------------------------------------

## 7. Side-by-Side Comparison: VAM vs Optimal

In [8]:
print("=" * 55)
print("       COMPARISON: VAM vs LINEAR PROGRAMMING")
print("=" * 55)
print()

comparison = pd.DataFrame({
    "VAM Allocation (kg)": vam_alloc.flatten(),
    "LP Allocation (kg)" : lp_alloc.flatten(),
}, index=[
    f"{sources[i]} -> {destinations[j]}"
    for i in range(m) for j in range(n)
])

# Show only routes used by at least one method
used = comparison[(comparison["VAM Allocation (kg)"] > 0) | (comparison["LP Allocation (kg)"] > 0)]
print(used.to_string())
print()

print(f"VAM Total Cost : KES {vam_cost:>10,}")
print(f"LP  Total Cost : KES {lp_cost:>10,}")
print()

if vam_cost == lp_cost:
    gap = 0
    print("Result: The VAM solution IS optimal.")
    print("The group's manual VAM matches the LP optimum exactly.")
else:
    gap = vam_cost - lp_cost
    pct = gap / lp_cost * 100
    print(f"Result: The VAM solution is NOT optimal.")
    print(f"Optimality gap : KES {gap:,}  ({pct:.2f}% above optimal)")
    print("The LP found a cheaper schedule. See the allocations above")
    print("to identify where the routes differ.")


       COMPARISON: VAM vs LINEAR PROGRAMMING

                                VAM Allocation (kg)  LP Allocation (kg)
Maua (S1) -> Isiolo (D1)                       1500                   0
Maua (S1) -> Kilifi (D2)                       1000                1500
Maua (S1) -> Likoni (D4)                          0                1000
Laare (S2) -> Eastleigh (D3)                   1800                1800
Kangeta (S3) -> Isiolo (D1)                       0                1500
Kangeta (S3) -> Kilifi (D2)                     500                   0
Kangeta (S3) -> Eastleigh (D3)                  200                 200
Kangeta (S3) -> Likoni (D4)                    1000                   0

VAM Total Cost : KES    101,700
LP  Total Cost : KES    101,700

Result: The VAM solution IS optimal.
The group's manual VAM matches the LP optimum exactly.


## 8. Sensitivity Check: Reduced Costs (MODI Method Verification)

For a basic feasible solution from VAM we can check optimality using the
Modified Distribution (MODI) method. If every unallocated cell has a
non-negative reduced cost (opportunity cost), the solution is optimal.
Below we compute reduced costs directly from the LP dual variables (shadow
prices), which is equivalent to MODI.


In [9]:
# Dual values from the LP
u = [prob.constraints[f"Supply_{i}"].pi for i in range(m)]
v = [prob.constraints[f"Demand_{j}"].pi for j in range(n)]

print("Dual variables (shadow prices)")
for i, src in enumerate(sources):
    print(f"  u[{i}] ({src:<15}) = {u[i]}")
for j, dst in enumerate(destinations):
    print(f"  v[{j}] ({dst:<20}) = {v[j]}")
print()

# Reduced costs for all cells
rc = np.array([[cost[i][j] - u[i] - v[j] for j in range(n)] for i in range(m)])
rc_df = pd.DataFrame(rc.round(2), index=sources, columns=destinations)
print("Reduced cost matrix (c[i][j] - u[i] - v[j])")
print("Allocated cells are shown as 0.0 by convention.")
print()

# Zero out allocated VAM cells for clarity
rc_display = rc_df.copy()
for i in range(m):
    for j in range(n):
        if lp_alloc[i, j] > 0:
            rc_display.iloc[i, j] = 0.0

print(rc_display.to_string())
print()
if (rc >= -1e-6).all():
    print("All reduced costs >= 0. The optimal solution is confirmed.")
else:
    neg = [(sources[i], destinations[j], round(rc[i,j],2))
           for i in range(m) for j in range(n) if rc[i,j] < -1e-6]
    print("Negative reduced costs found (these cells could improve the solution):")
    for entry in neg:
        print(f"  {entry[0]} -> {entry[1]}: {entry[2]}")


Dual variables (shadow prices)
  u[0] (Maua (S1)      ) = 10.0
  u[1] (Laare (S2)     ) = 9.0
  u[2] (Kangeta (S3)   ) = 10.0
  v[0] (Isiolo (D1)         ) = 0.0
  v[1] (Kilifi (D2)         ) = 11.0
  v[2] (Eastleigh (D3)      ) = 7.0
  v[3] (Likoni (D4)         ) = 13.0

Reduced cost matrix (c[i][j] - u[i] - v[j])
Allocated cells are shown as 0.0 by convention.

              Isiolo (D1)  Kilifi (D2)  Eastleigh (D3)  Likoni (D4)
Maua (S1)             0.0          0.0             1.0          0.0
Laare (S2)            2.0          2.0             0.0          1.0
Kangeta (S3)          0.0          0.0             0.0          0.0

All reduced costs >= 0. The optimal solution is confirmed.


## 9. Summary

In [10]:
print("FINAL SUMMARY")
print("=" * 50)
print()
print("Problem  : Transportation of Miraa, Meru County")
print("Sources  : Maua (2,500 kg), Laare (1,800 kg), Kangeta (1,700 kg)")
print("Markets  : Isiolo (1,500 kg), Kilifi (1,500 kg),")
print("           Eastleigh (2,000 kg), Likoni (1,000 kg)")
print("Balanced : Yes. Total supply = total demand = 6,000 kg")
print()
print(f"VAM Cost : KES {vam_cost:,}")
print(f"LP  Cost : KES {lp_cost:,}")
gap = vam_cost - lp_cost
if gap == 0:
    print()
    print("Conclusion: VAM is OPTIMAL for this problem.")
    print("The manual solution in the group report is confirmed correct.")
else:
    pct = gap / lp_cost * 100
    print(f"Gap      : KES {gap:,} ({pct:.2f}% above optimal)")
    print()
    print("Conclusion: VAM is SUBOPTIMAL by the gap shown above.")
    print("Review the allocation table in Section 7 for the better routes.")


FINAL SUMMARY

Problem  : Transportation of Miraa, Meru County
Sources  : Maua (2,500 kg), Laare (1,800 kg), Kangeta (1,700 kg)
Markets  : Isiolo (1,500 kg), Kilifi (1,500 kg),
           Eastleigh (2,000 kg), Likoni (1,000 kg)
Balanced : Yes. Total supply = total demand = 6,000 kg

VAM Cost : KES 101,700
LP  Cost : KES 101,700

Conclusion: VAM is OPTIMAL for this problem.
The manual solution in the group report is confirmed correct.
